In [0]:
from pyspark.sql import functions as F
df = spark.table("workspace.default.cfpb_complaints_silver")
gold_monthly = (df.withColumn("complaint_month", F.trunc("date_received","month"))
                .groupBy("complaint_month", "product_5")
                .agg(F.countDistinct("complaint_id").alias("distinct_complaints"))
)
gold_monthly.orderBy("complaint_month", "product_5").show(20, truncate=False)


+---------------+--------------------------+-------------------+
|complaint_month|product_5                 |distinct_complaints|
+---------------+--------------------------+-------------------+
|2015-03-01     |Bank Accounts and Services|202                |
|2015-03-01     |Credit Card Services      |252                |
|2015-03-01     |Credit Reporting          |370                |
|2015-03-01     |Debt Collection           |762                |
|2015-03-01     |Loans                     |791                |
|2015-04-01     |Bank Accounts and Services|477                |
|2015-04-01     |Credit Card Services      |607                |
|2015-04-01     |Credit Reporting          |904                |
|2015-04-01     |Debt Collection           |1561               |
|2015-04-01     |Loans                     |1830               |
|2015-05-01     |Bank Accounts and Services|483                |
|2015-05-01     |Credit Card Services      |729                |
|2015-05-01     |Credit R

In [0]:
gold_monthly.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_complaints_by_month_product")

gold_saved = spark.table("workspace.default.gold_complaints_by_month_product")
gold_saved.agg(F.sum("distinct_complaints")).show()

+------------------------+
|sum(distinct_complaints)|
+------------------------+
|                 2023066|
+------------------------+



In [0]:
from pyspark.sql import functions as F
df = spark.table("workspace.default.cfpb_complaints_silver")

gold_company = (
    df.groupBy("company")
    .agg(F.countDistinct("complaint_id").alias("distinct_complaints"))
    .orderBy(F.desc("distinct_complaints"))
)

gold_company.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_complaints_by_company")

gold_company.show(10, truncate=False)

+--------------------------------------+-------------------+
|company                               |distinct_complaints|
+--------------------------------------+-------------------+
|EQUIFAX, INC.                         |372385             |
|TRANSUNION INTERMEDIATE HOLDINGS, INC.|362238             |
|Experian Information Solutions Inc.   |337567             |
|CAPITAL ONE FINANCIAL CORPORATION     |44106              |
|WELLS FARGO & COMPANY                 |44040              |
|JPMORGAN CHASE & CO.                  |43842              |
|BANK OF AMERICA, NATIONAL ASSOCIATION |42390              |
|CITIBANK, N.A.                        |38986              |
|SYNCHRONY FINANCIAL                   |23857              |
|Navient Solutions, LLC.               |19224              |
+--------------------------------------+-------------------+
only showing top 10 rows


In [0]:
from pyspark.sql import functions as F
gold_by_company = spark.table("workspace.default.gold_complaints_by_company")
gold_by_company.agg(F.sum("distinct_complaints").alias("total_complaints")).show()

+----------------+
|total_complaints|
+----------------+
|         2023066|
+----------------+



In [0]:
gold_by_company.filter(F.lower("company").contains("equifax")).show(truncate=False)

+-------------+-------------------+
|company      |distinct_complaints|
+-------------+-------------------+
|EQUIFAX, INC.|372385             |
+-------------+-------------------+



In [0]:
gold_by_company.filter(F.lower("company").contains("transunion")).show(truncate=False)
gold_by_company.filter(F.lower("company").contains("experian")).show(truncate=False)

+--------------------------------------+-------------------+
|company                               |distinct_complaints|
+--------------------------------------+-------------------+
|TRANSUNION INTERMEDIATE HOLDINGS, INC.|362238             |
+--------------------------------------+-------------------+

+-----------------------------------+-------------------+
|company                            |distinct_complaints|
+-----------------------------------+-------------------+
|Experian Information Solutions Inc.|337567             |
+-----------------------------------+-------------------+



In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.default.cfpb_complaints_silver")

gold_company_product = (
    df.groupBy("company", "product_5")
    .agg(F.countDistinct("complaint_id").alias("distinct_complaints"))
    .orderBy(F.desc("distinct_complaints"))
)

gold_company_product.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_complaints_by_company_product")

spark.table("workspace.default.gold_complaints_by_company_product") \
    .agg(F.sum("distinct_complaints")).show()

+------------------------+
|sum(distinct_complaints)|
+------------------------+
|                 2023066|
+------------------------+



In [0]:
display(spark.table("workspace.default.gold_complaints_by_company_product"))

company,product_5,distinct_complaints
"EQUIFAX, INC.",Credit Reporting,360826
"TRANSUNION INTERMEDIATE HOLDINGS, INC.",Credit Reporting,348937
Experian Information Solutions Inc.,Credit Reporting,325621
"CITIBANK, N.A.",Credit Card Services,20088
CAPITAL ONE FINANCIAL CORPORATION,Credit Card Services,19301
WELLS FARGO & COMPANY,Bank Accounts and Services,18501
"BANK OF AMERICA, NATIONAL ASSOCIATION",Bank Accounts and Services,18108
JPMORGAN CHASE & CO.,Bank Accounts and Services,17468
"Navient Solutions, LLC.",Loans,15930
WELLS FARGO & COMPANY,Loans,13913


In [0]:
spark.table("workspace.default.gold_complaints_by_company_product").count()

11916